# Model Training — Cultural Pattern Discovery

**Repo:** [cultural-pattern-discovery-ml](https://github.com/ishangupta4/cultural-pattern-discovery-ml)  
**Branch:** `feature/feature-engineering`  
**Upstream:** EDA + feature engineering already done in `met_eda.ipynb` and `02_preprocessing.ipynb`

This notebook **only** does model training. It loads the preprocessed sparse matrices and label encoder produced by the existing pipeline, then runs:

1. Supervised baselines (6 classifiers)
2. Handling class imbalance
3. Hyperparameter tuning on top performers
4. Cross-validation of the final model
5. Unsupervised clustering (KMeans) for cultural pattern discovery
6. Evaluation, plots, and saving artifacts

---

## 0. Clone repo & regenerate preprocessed splits

In [ ]:
import os

REPO_DIR = '/content/cultural-pattern-discovery-ml'

if not os.path.exists(REPO_DIR):
    !git clone -b feature/feature-engineering \
        https://github.com/ishangupta4/cultural-pattern-discovery-ml.git {REPO_DIR}
else:
    print('Repo already cloned.')

os.chdir(REPO_DIR)
!pwd

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# Download the raw dataset (needed for the preprocessing notebook)
DATA_PATH = 'data/MetObjects.csv'
os.makedirs('data/processed', exist_ok=True)

if not os.path.exists(DATA_PATH):
    print('Downloading MetObjects.csv ...')
    !curl -L "https://media.githubusercontent.com/media/metmuseum/openaccess/master/MetObjects.csv" \
        -o {DATA_PATH}
else:
    print(f'Dataset already at {DATA_PATH}')

In [ ]:
# The processed splits (data/processed/*.npz, *.npy) are git-ignored.
# Regenerate them by running the preprocessing notebook.

SPLITS_EXIST = all(
    os.path.exists(f'data/processed/{f}')
    for f in ['X_train.npz', 'X_test.npz', 'y_train.npy', 'y_test.npy']
)

if not SPLITS_EXIST:
    print('Running 02_preprocessing.ipynb to generate train/test splits...')
    !pip install -q nbconvert
    !jupyter nbconvert --to notebook --execute notebooks/02_preprocessing.ipynb \
        --ExecutePreprocessor.timeout=900 \
        --output /tmp/preprocessing_executed.ipynb 2>&1 | tail -5
    print('Preprocessing complete.')
else:
    print('Preprocessed splits already exist — skipping.')

## 1. Load preprocessed data

In [ ]:
import numpy as np
import joblib
from scipy.sparse import load_npz

X_train = load_npz('data/processed/X_train.npz')   # ~388k rows, ~215 cols, sparse
X_test  = load_npz('data/processed/X_test.npz')     # ~97k rows
y_train = np.load('data/processed/y_train.npy')     # integer labels
y_test  = np.load('data/processed/y_test.npy')

le = joblib.load('models/label_encoder.joblib')     # Department ↔ int mapping

print(f'X_train : {X_train.shape}  (sparse, {X_train.dtype})')
print(f'X_test  : {X_test.shape}')
print(f'y_train : {y_train.shape}   |  classes: {len(le.classes_)}')
print(f'y_test  : {y_test.shape}')
print(f'\nDepartments ({len(le.classes_)}):\n{le.classes_}')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import warnings, time

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

# Class distribution
unique, counts = np.unique(y_train, return_counts=True)
dept_counts = pd.Series(counts, index=le.inverse_transform(unique)).sort_values()

fig, ax = plt.subplots(figsize=(12, 6))
dept_counts.plot(kind='barh', color='steelblue', ax=ax)
ax.set_xlabel('Training samples')
ax.set_title('Class distribution — 21 departments (heavily imbalanced)')
plt.tight_layout()
plt.show()

print(f'Imbalance ratio (max/min): {counts.max()/counts.min():.0f}x')

---
## 2. Supervised baselines

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    f1_score, ConfusionMatrixDisplay
)

models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, solver='saga', multi_class='multinomial',
        random_state=42, n_jobs=-1
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=20, min_samples_leaf=5, random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=25, min_samples_leaf=3,
        random_state=42, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, random_state=42
    ),
    'KNN': KNeighborsClassifier(n_neighbors=7, n_jobs=-1),
    'Linear SVC': LinearSVC(max_iter=2000, random_state=42),
}

print(f'{len(models)} models queued')

In [ ]:
results = []

for name, clf in models.items():
    print(f'\n{"="*55}')
    print(f' Training: {name}')
    print(f'{"="*55}')

    t0 = time.time()
    clf.fit(X_train, y_train)
    train_sec = time.time() - t0

    t0 = time.time()
    y_pred = clf.predict(X_test)
    pred_sec = time.time() - t0

    acc  = accuracy_score(y_test, y_pred)
    f1_w = f1_score(y_test, y_pred, average='weighted')
    f1_m = f1_score(y_test, y_pred, average='macro')

    results.append(dict(
        model=name, clf=clf,
        accuracy=acc, f1_weighted=f1_w, f1_macro=f1_m,
        train_s=round(train_sec, 2), pred_s=round(pred_sec, 2)
    ))

    print(f'  Accuracy    : {acc:.4f}')
    print(f'  F1 macro    : {f1_m:.4f}   <- primary metric')
    print(f'  F1 weighted : {f1_w:.4f}')
    print(f'  Time (train): {train_sec:.1f}s   (predict): {pred_sec:.2f}s')

print('\nAll baselines trained.')

In [ ]:
res_df = (
    pd.DataFrame([{k:v for k,v in r.items() if k!='clf'} for r in results])
      .sort_values('f1_macro', ascending=False)
      .reset_index(drop=True)
)
res_df.style.background_gradient(
    subset=['accuracy','f1_weighted','f1_macro'], cmap='Greens'
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

melted = res_df.melt(
    id_vars='model',
    value_vars=['accuracy','f1_weighted','f1_macro'],
    var_name='metric', value_name='score'
)
sns.barplot(data=melted, x='score', y='model', hue='metric', ax=axes[0])
axes[0].set_xlim(0,1); axes[0].set_title('Classification metrics')

sns.barplot(data=res_df, x='train_s', y='model', color='coral', ax=axes[1])
axes[1].set_title('Training time (seconds)')

plt.tight_layout(); plt.show()

In [ ]:
# Best baseline — full report
best = max(results, key=lambda r: r['f1_macro'])
print(f'Best baseline: {best["model"]}  (macro F1 = {best["f1_macro"]:.4f})\n')

y_pred_best = best['clf'].predict(X_test)
print(classification_report(
    y_test, y_pred_best,
    target_names=le.classes_, zero_division=0
))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 12))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_best,
    display_labels=le.classes_, ax=ax,
    cmap='Blues', xticks_rotation=45, values_format='d'
)
ax.set_title(f'Confusion matrix — {best["model"]}')
plt.tight_layout(); plt.show()

---
## 3. Handling class imbalance

21 departments are heavily imbalanced. Two strategies:
- `class_weight='balanced'` (re-weights the loss)
- SMOTE oversampling on minority classes

In [ ]:
balanced_models = {
    'LogReg (balanced)': LogisticRegression(
        max_iter=1000, solver='saga', multi_class='multinomial',
        class_weight='balanced', random_state=42, n_jobs=-1
    ),
    'RF (balanced)': RandomForestClassifier(
        n_estimators=200, max_depth=25, min_samples_leaf=3,
        class_weight='balanced', random_state=42, n_jobs=-1
    ),
    'LinearSVC (balanced)': LinearSVC(
        max_iter=2000, class_weight='balanced', random_state=42
    ),
}

for name, clf in balanced_models.items():
    t0 = time.time()
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    f1_m = f1_score(y_test, y_pred, average='macro')
    f1_w = f1_score(y_test, y_pred, average='weighted')
    acc  = accuracy_score(y_test, y_pred)
    results.append(dict(
        model=name, clf=clf,
        accuracy=acc, f1_weighted=f1_w, f1_macro=f1_m,
        train_s=round(time.time()-t0,2), pred_s=None
    ))
    print(f'{name:25s}  macro-F1={f1_m:.4f}  weighted-F1={f1_w:.4f}  acc={acc:.4f}')

In [ ]:
# SMOTE oversampling
!pip install -q imbalanced-learn
from imblearn.over_sampling import SMOTE

SMOTE_CAP = 80_000
rng = np.random.RandomState(42)
if X_train.shape[0] > SMOTE_CAP:
    idx = rng.choice(X_train.shape[0], SMOTE_CAP, replace=False)
    X_sm, y_sm = X_train[idx], y_train[idx]
    print(f'SMOTE on {SMOTE_CAP:,}-sample subset')
else:
    X_sm, y_sm = X_train, y_train

smote = SMOTE(random_state=42, k_neighbors=3)
X_res, y_res = smote.fit_resample(X_sm, y_sm)
print(f'After SMOTE: {X_res.shape[0]:,} samples (was {X_sm.shape[0]:,})')

clf_smote = RandomForestClassifier(
    n_estimators=200, max_depth=25, min_samples_leaf=3,
    random_state=42, n_jobs=-1
)
clf_smote.fit(X_res, y_res)
y_pred_smote = clf_smote.predict(X_test)

f1_m = f1_score(y_test, y_pred_smote, average='macro')
f1_w = f1_score(y_test, y_pred_smote, average='weighted')
acc  = accuracy_score(y_test, y_pred_smote)
print(f'RF+SMOTE:  macro-F1={f1_m:.4f}  weighted-F1={f1_w:.4f}  acc={acc:.4f}')

results.append(dict(
    model='RF + SMOTE', clf=clf_smote,
    accuracy=acc, f1_weighted=f1_w, f1_macro=f1_m, train_s=None, pred_s=None
))

---
## 4. Hyperparameter tuning

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, GridSearchCV

# Subsample for speed on Colab free tier
TUNE_CAP = 60_000
if X_train.shape[0] > TUNE_CAP:
    idx = rng.choice(X_train.shape[0], TUNE_CAP, replace=False)
    X_tune, y_tune = X_train[idx], y_train[idx]
else:
    X_tune, y_tune = X_train, y_train

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
print(f'Tuning sample: {X_tune.shape[0]:,} rows')

In [ ]:
# Random Forest tuning
rf_params = {
    'n_estimators':     [100, 200, 400],
    'max_depth':        [15, 25, 40, None],
    'min_samples_leaf': [1, 3, 5],
    'max_features':     ['sqrt', 'log2'],
    'class_weight':     ['balanced', 'balanced_subsample', None],
}

print('RandomizedSearchCV (RF) — 30 iters, 3-fold...')
rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    rf_params, n_iter=30, cv=cv,
    scoring='f1_macro', random_state=42, n_jobs=-1, verbose=1
)
rf_search.fit(X_tune, y_tune)

print(f'\nBest CV macro-F1: {rf_search.best_score_:.4f}')
print(f'Best params: {rf_search.best_params_}')

In [ ]:
# Logistic Regression tuning
lr_params = {
    'C':            [0.01, 0.1, 1, 10],
    'class_weight': ['balanced', None],
}

print('GridSearchCV (LogReg) — 3-fold...')
lr_search = GridSearchCV(
    LogisticRegression(max_iter=1000, solver='saga',
                       multi_class='multinomial', random_state=42, n_jobs=-1),
    lr_params, cv=cv, scoring='f1_macro', n_jobs=-1, verbose=1
)
lr_search.fit(X_tune, y_tune)

print(f'\nBest CV macro-F1: {lr_search.best_score_:.4f}')
print(f'Best params: {lr_search.best_params_}')

In [ ]:
# LinearSVC tuning
svc_params = {
    'C':            [0.01, 0.1, 1, 10],
    'class_weight': ['balanced', None],
}

print('GridSearchCV (LinearSVC) — 3-fold...')
svc_search = GridSearchCV(
    LinearSVC(max_iter=3000, random_state=42),
    svc_params, cv=cv, scoring='f1_macro', n_jobs=-1, verbose=1
)
svc_search.fit(X_tune, y_tune)

print(f'\nBest CV macro-F1: {svc_search.best_score_:.4f}')
print(f'Best params: {svc_search.best_params_}')

In [ ]:
# Retrain tuned models on FULL training set
tuned = {
    'RF (tuned)':        rf_search.best_estimator_,
    'LogReg (tuned)':    lr_search.best_estimator_,
    'LinearSVC (tuned)': svc_search.best_estimator_,
}

for name, clf in tuned.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    f1_m = f1_score(y_test, y_pred, average='macro')
    f1_w = f1_score(y_test, y_pred, average='weighted')
    acc  = accuracy_score(y_test, y_pred)
    results.append(dict(
        model=name, clf=clf,
        accuracy=acc, f1_weighted=f1_w, f1_macro=f1_m,
        train_s=None, pred_s=None
    ))
    print(f'{name:22s}  macro-F1={f1_m:.4f}  weighted-F1={f1_w:.4f}  acc={acc:.4f}')

---
## 5. Cross-validation of final model

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.base import clone

overall_best = max(results, key=lambda r: r['f1_macro'])
print(f'Overall best: {overall_best["model"]}  (test macro-F1={overall_best["f1_macro"]:.4f})')

CV_CAP = 60_000
if X_train.shape[0] > CV_CAP:
    idx = rng.choice(X_train.shape[0], CV_CAP, replace=False)
    X_cv, y_cv = X_train[idx], y_train[idx]
else:
    X_cv, y_cv = X_train, y_train

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_clf = clone(overall_best['clf'])

print(f'5-fold CV on {X_cv.shape[0]:,} samples...')
cv_scores = cross_val_score(cv_clf, X_cv, y_cv, cv=skf, scoring='f1_macro', n_jobs=-1)

print(f'Fold scores: {cv_scores}')
print(f'Mean macro-F1: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')

---
## 6. Full leaderboard & best-model deep-dive

In [ ]:
leader = (
    pd.DataFrame([{k:v for k,v in r.items() if k!='clf'} for r in results])
      .sort_values('f1_macro', ascending=False)
      .reset_index(drop=True)
)
leader.style.background_gradient(
    subset=['accuracy','f1_weighted','f1_macro'], cmap='Greens'
)

In [ ]:
best_clf = overall_best['clf']
y_final = best_clf.predict(X_test)

print(f'=== {overall_best["model"]} — classification report ===\n')
print(classification_report(
    y_test, y_final,
    target_names=le.classes_, zero_division=0
))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 12))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_final,
    display_labels=le.classes_, ax=ax,
    cmap='Blues', xticks_rotation=45, values_format='d'
)
ax.set_title(f'Confusion matrix — {overall_best["model"]}')
plt.tight_layout(); plt.show()

In [ ]:
# Per-class F1 (spot weak departments)
per_class = f1_score(y_test, y_final, average=None, zero_division=0)
pc = pd.DataFrame({'department': le.classes_, 'f1': per_class}).sort_values('f1')

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#d9534f' if v < 0.3 else '#5cb85c' for v in pc['f1']]
ax.barh(pc['department'], pc['f1'], color=colors)
ax.axvline(per_class.mean(), color='navy', ls='--', label=f'mean={per_class.mean():.3f}')
ax.set_xlabel('F1'); ax.set_title(f'Per-class F1 — {overall_best["model"]}')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Feature importance (tree-based models)
if hasattr(best_clf, 'feature_importances_'):
    pipeline = joblib.load('models/preprocessing_pipeline.joblib')
    try:
        feat_names = pipeline.get_feature_names_out()
    except Exception:
        feat_names = [f'f{i}' for i in range(X_train.shape[1])]

    imp = pd.Series(best_clf.feature_importances_, index=feat_names)
    top30 = imp.sort_values(ascending=False).head(30)

    fig, ax = plt.subplots(figsize=(10, 9))
    top30.plot(kind='barh', ax=ax, color='steelblue')
    ax.invert_yaxis()
    ax.set_title(f'Top 30 features — {overall_best["model"]}')
    ax.set_xlabel('Importance')
    plt.tight_layout(); plt.show()
else:
    print(f'{overall_best["model"]} has no feature_importances_.')

---
## 7. Unsupervised — Cultural pattern discovery

In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

# TruncatedSVD works directly on sparse matrices (no densification needed)
CLUSTER_CAP = 40_000
idx = rng.choice(X_train.shape[0], min(CLUSTER_CAP, X_train.shape[0]), replace=False)
X_cl = X_train[idx]
y_cl = y_train[idx]

svd = TruncatedSVD(n_components=50, random_state=42)
X_svd = svd.fit_transform(X_cl)
print(f'SVD: {X_cl.shape} -> {X_svd.shape}')
print(f'Explained variance (50 comps): {svd.explained_variance_ratio_.sum():.2%}')

In [ ]:
# Elbow + silhouette
K_range = range(5, 31, 2)
inertias, sils = [], []

for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_svd)
    inertias.append(km.inertia_)
    s = silhouette_score(X_svd, labels, sample_size=5000, random_state=42)
    sils.append(s)
    print(f'K={k:2d}  silhouette={s:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(K_range), inertias, 'o-')
axes[0].set_title('Elbow'); axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia')
axes[1].plot(list(K_range), sils, 'o-', color='green')
axes[1].set_title('Silhouette'); axes[1].set_xlabel('K'); axes[1].set_ylabel('Score')
plt.tight_layout(); plt.show()

In [ ]:
best_k = list(K_range)[np.argmax(sils)]
print(f'Best K by silhouette: {best_k}')

kmeans = KMeans(n_clusters=best_k, n_init=20, random_state=42)
cluster_labels = kmeans.fit_predict(X_svd)

sil = silhouette_score(X_svd, cluster_labels, sample_size=5000, random_state=42)
ari = adjusted_rand_score(y_cl, cluster_labels)
print(f'Silhouette: {sil:.4f}')
print(f'ARI vs Department: {ari:.4f}')

In [ ]:
# t-SNE visualization
from sklearn.manifold import TSNE

TSNE_N = min(8000, X_svd.shape[0])
t_idx = rng.choice(X_svd.shape[0], TSNE_N, replace=False)

print(f't-SNE on {TSNE_N:,} points...')
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_2d = tsne.fit_transform(X_svd[t_idx])

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

axes[0].scatter(X_2d[:,0], X_2d[:,1], c=cluster_labels[t_idx], cmap='tab20', s=2, alpha=.6)
axes[0].set_title(f't-SNE — KMeans clusters (K={best_k})')

axes[1].scatter(X_2d[:,0], X_2d[:,1], c=y_cl[t_idx], cmap='tab20', s=2, alpha=.6)
axes[1].set_title('t-SNE — actual departments')

plt.tight_layout(); plt.show()

In [ ]:
# Cluster x Department heatmap
ct = pd.crosstab(
    pd.Series(cluster_labels, name='Cluster'),
    pd.Series(le.inverse_transform(y_cl), name='Department'),
    normalize='index'
)

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(ct, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax, linewidths=.5)
ax.set_title('Cluster composition by department (row-normalized)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()

In [ ]:
# Cluster profiling
dept_names = le.inverse_transform(y_cl)
print('='*55)
print('Cluster -> top departments')
print('='*55)
for c in range(best_k):
    mask = cluster_labels == c
    total = mask.sum()
    vals, cnts = np.unique(dept_names[mask], return_counts=True)
    order = np.argsort(-cnts)
    print(f'\nCluster {c}  ({total:,} items)')
    for i in order[:3]:
        print(f'  {vals[i]}: {cnts[i]:,} ({cnts[i]/total:.1%})')

---
## 8. Save artifacts

In [ ]:
os.makedirs('models', exist_ok=True)
os.makedirs('reports', exist_ok=True)

tag = overall_best['model'].replace(' ','_').replace('(','').replace(')','').lower()

joblib.dump(best_clf, f'models/best_classifier_{tag}.joblib')
print(f'Saved: models/best_classifier_{tag}.joblib')

joblib.dump({'svd': svd, 'kmeans': kmeans, 'best_k': best_k},
            'models/clustering_artifacts.joblib')
print('Saved: models/clustering_artifacts.joblib')

leader.to_csv('reports/model_comparison.csv', index=False)
print('Saved: reports/model_comparison.csv')

print('\nAll artifacts saved.')

In [ ]:
# Optional: push to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp models/best_classifier_*.joblib models/clustering_artifacts.joblib \
#      reports/model_comparison.csv /content/drive/MyDrive/

---
## Summary

### What this notebook did
- Loaded **preprocessed sparse matrices** (`X_train.npz`, `X_test.npz`, ~215 features) and **label encoder** from the `feature/feature-engineering` branch — zero duplicate preprocessing
- Trained **6 baseline classifiers**, compared on **macro F1** (primary metric for 21 imbalanced departments)
- Addressed **class imbalance** with `class_weight='balanced'` and **SMOTE**
- **Hyperparameter tuning** via RandomizedSearchCV / GridSearchCV for RF, LogReg, LinearSVC
- **5-fold stratified CV** on the overall best model
- **Per-class F1 analysis** to spot weak departments
- **KMeans clustering** on TruncatedSVD-reduced features for latent cultural pattern discovery
- Cluster profiling via heatmap, t-SNE, and department composition

### Next steps
- Try **XGBoost / LightGBM** (handles sparse matrices + imbalance natively)
- **SHAP** analysis for model interpretability
- **Hierarchical clustering** + dendrogram for sub-cultural groupings
- **Error analysis**: inspect misclassified samples from weak departments